# Clinical RAG Assistant — Demo Walkthrough

This notebook runs the real pipeline end-to-end (real embeddings, real
generation) against the bundled synthetic sample guideline.

**Requires network access** to download `sentence-transformers/all-MiniLM-L6-v2`
and `Qwen/Qwen2.5-1.5B-Instruct` from Hugging Face on first run. This notebook is a
portfolio/demo artifact — it is intentionally **not** part of the automated
(offline) test suite or CI, which use dependency-injected fakes instead.

In [ ]:
from src.ingestion.document_loader import load_document
from src.ingestion.chunker import chunk_documents
from src.embeddings.embedder import ClinicalEmbedder
from src.retrieval.vector_store import FAISSVectorStore
from src.generation.rag_chain import RAGChain

## 1. Load and chunk the sample document

In [ ]:
docs = load_document("../examples/sample_docs/diabetes_management_guideline.md")
chunks = chunk_documents(docs)
print(f"{len(chunks)} chunks produced")
for chunk in chunks[:3]:
    print("---")
    print(chunk.text)

## 2. Embed and index the chunks

In [ ]:
embedder = ClinicalEmbedder()  # downloads all-MiniLM-L6-v2 on first use
embeddings = embedder.encode([c.text for c in chunks])

store = FAISSVectorStore(dim=embeddings.shape[1])
store.add(
    embeddings,
    texts=[c.text for c in chunks],
    sources=[c.source for c in chunks],
    pages=[c.page for c in chunks],
)
print(f"Indexed {len(store)} chunks")

## 3. Ask a question

The generator (`Qwen/Qwen2.5-1.5B-Instruct`) downloads on first use here too.

In [ ]:
chain = RAGChain(embedder, store)
response = chain.query("What is the target HbA1c for most adults with type 2 diabetes?")

print(response.answer)
print("\nGrounded:", response.grounded)
print("\nSources:")
for source in response.sources:
    print(f"  - {source.source} (page {source.page}, score {source.score:.2f})")

## 4. An out-of-scope question

Demonstrates the explicit insufficient-information fallback rather than a
fabricated answer.

In [ ]:
response = chain.query("What is the capital of France?")
print(response.answer)
print("\nGrounded:", response.grounded)